In [4]:
# Exercise 10 : Metadata Aware RAG : 

from pypdf import PdfReader
pdf_files = ["os.pdf", "dbms.pdf", "oops.pdf", "computer_networks.pdf"]
documents = []


In [5]:
def chunk_text(text, chunk_size=100, overlap=20):

    words = text.split()
    step = chunk_size - overlap
    chunks = []
    for i in range(0,len(words),step):
        chunk = " ".join(words[i:i+chunk_size])

        if chunk:
            chunks.append(chunk)
    return chunks


In [6]:
for pdf_file in pdf_files:
    reader = PdfReader(pdf_file)

    for page_num, page in enumerate(reader.pages, start = 1):
        text = page.extract_text()
        if not text:
            continue
        page_chunks = chunk_text(text)
        for chunk in page_chunks:
            documents.append(
                {
                    "text":chunk,
                    "page":page_num,
                    "source":pdf_file
                }
            )


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

chunk_texts = [
    doc["text"]
    for doc in documents
]

chunk_embeddings = model.encode(chunk_texts)

W0629 18:58:58.872000 52464 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
C:\Users\uniqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
import numpy as np
chunk_embeddings = np.array(
    chunk_embeddings,
    dtype=np.float32
)

In [9]:
import faiss
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)


In [10]:
def retrieve(query, k=3):
    query_embedding = model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    distances, indices = index.search(
        query_embedding,
        k
    )

    return [documents[i] for i in indices[0]]


In [11]:
def build_prompt(query, retrieved_chunks, chat_history):

    context = ""

    for doc in retrieved_chunks:

        context += (
            f"Source: {doc['source']}, "
            f"Page: {doc['page']}\n"
        )

        context += doc["text"] + "\n\n"

    history = "\n".join(chat_history)

    prompt = f"""
Context:
{context}

History:
{history}

Question:
{query}

Answer using only the context and history.
"""

    return prompt

In [ ]:
from google import genai

client = genai.Client(api_key="GEMINI_API_KEY")


def rewrite_query(query, recent_questions):
    history = " ".join(recent_questions)
    rewrite_prompt = f"""
Given a conversation history and current question, rewrite the current question
into a standalone question.
History : 
{history}

Current Question : 
{query}

Standalone Question : 
"""
    response = client.models.generate_content(
        model = "gemini-2.5-flash",
        contents = rewrite_prompt
    )

    return response.text.strip()


In [13]:
chat_history = []
recent_questions = []

while True:

    query = input("Ask: ")

    if query == "exit":
        break

    standalone_query = rewrite_query(query, recent_questions)
    results = retrieve(standalone_query,3)
    
    print("Standalone Query : ")
    print(standalone_query)

    prompt = build_prompt(
        query,
        results,
        chat_history
    )
    for result in results:

        print(
            f"\nSource: {result['source']}"
        )

        print(
            f"Page: {result['page']}"
        )

        print(
            result["text"]
        )

    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
    )

    print("Answer : ")
    print(response.text)

    chat_history.append(
        f"User: {query}"
    )

    chat_history.append(
        f"Assistant: {response.text}"
    )

    recent_questions.append(query)
    recent_questions = recent_questions[-3:]
    chat_history = chat_history[-6:]

Ask:  what is paging?


Standalone Query : 
what is paging?

Source: os.pdf
Page: 18
and the rest on disk. Demand paging loads a page only when it is first accessed. If the accessed page is not in memory, the hardware raises a page fault. The OS finds the page on disk, loads it into a free frame (or evicts one via a page-replacement algorithm), updates the page table, and restarts the instruction. 💡 Aha - Virtual memory aise hai jaise tum poori library ghar nahi laate - jo kitaab abhi chahiye wahi shelf (RAM) pe rakhte ho, baaki godown (disk) mein. Page fault tab hota hai jab zaroori kitaab shelf pe nahi mili, toh

Source: os.pdf
Page: 26
That swap is the single most asked follow-up here. Q. What is virtual memory and how does demand paging work? Say this: Virtual memory is the illusion that each process has a large, contiguous address space even when physical RAM is smaller, by keeping only the pages it currently needs in memory and the rest on disk. Demand paging is how it loads them: a page is brought in o

Ask:  what is indexing?


Standalone Query : 
what is indexing?

Source: dbms.pdf
Page: 16
@rathoreadiitya DBMS Mastery v1.0 Page 16 of 27 How each level actually does it: • Read Uncommitted - no read locks, so it prevents nothing. • Read Committed - a read only sees committed values, killing dirty reads. • Repeatable Read - holds read locks on rows it touched until commit, so a re-read is stable (but new rows can still appear). • Serializable - locks the query range itself, so even phantom inserts are blocked. 20. Indexing An index is a side structure that lets the DBMS find rows without scanning the whole table - like the index at

Source: dbms.pdf
Page: 16
index-key order, so there is at most ONE per table (the rows can only be sorted one way). A non-clustered index is a separate structure holding the key plus a pointer to the row; you can have many per table.

Source: dbms.pdf
Page: 3
of the four letters guarantees, with a one-line example. 3. The isolation grid - which of dirty read, non-repeatable read, a

Ask:  Are these two related?


Standalone Query : 
Are paging and indexing related?

Source: os.pdf
Page: 17
address splits into a page number (index into the table) and an offset (copied straight through). Paging kills external fragmentation but can leave a little internal fragmentation in the last page. Logical address = (page, offset). The page table turns page into frame, the offset never changes. The TLB caches recent lookups. 💡 Aha - Paging cloud storage jaisa hai - tumhari file (process) chhote chhote fixed chunks (pages) mein toot jaati hai aur RAM ke kisi bhi khaali locker (frame) mein rakhi ja sakti hai. Page table woh index hai jo batata hai kaunsa chunk kis locker mein pada

Source: os.pdf
Page: 18
@rathoreadiitya OS Mastery v1.0 Page 18 of 31 Segmentation Segmentation splits memory by logical units - code, stack, heap - of variable size, matching how a programmer thinks. An address is (segment number, offset). It gives a natural view but brings back external fragmentation. Paging vs segmentation is a cl

Ask:  exit
